# Houseplant SFT v2 — Colab T4: обучение + оценка

Один ноутбук: установка → pilot → полное QLoRA на `data/sft_v2/sft_train.jsonl` (1 060 пар, отбор судьёй, см. `reports/sft_v2_data.md`) → адаптер на Drive → base vs SFT на plant + garden + general, судья, правило выбора чекпоинта, test, графики.

Стек (pinned): `transformers==4.57.1` + `trl==0.24.0` + `peft==0.17.1` + `bitsandbytes` (Linux).

1. Runtime → Change runtime type → **T4 GPU**.
2. Secrets (ключ слева) → `OPENROUTER_API_KEY`, включить Notebook access.
3. Runtime → **Run all**. Вкладку не закрывать.

Повторный Run all безопасен: если адаптер уже на Drive (`MyDrive/rk1_houseplant/s2_sft_v2`), обучение пропускается; оценка пишется на Drive построчно и продолжает с места остановки.

- `SMOKE = True` — оценка по 3 примера на набор в отдельную папку (обучение не меняется).
- `FORCE_TRAIN = True` — переобучить, даже если адаптер на Drive уже есть.
- Старые `eval_s1` / `eval_smoke` не переиспользовать: другой системный промпт, база там оценена по-старому.
- Если после старого `--force-reinstall` сломан numpy/torch: Runtime → Disconnect and delete runtime. Не используйте `pip install -U` / `--force-reinstall` на весь файл требований.

In [2]:
from google.colab import drive, userdata
import os

SMOKE = False
FORCE_TRAIN = False

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rk1_houseplant'
ADAPTER = f'{DRIVE}/s2_sft_v2'
OUT = f'{DRIVE}/eval_smoke_s2' if SMOKE else f'{DRIVE}/eval_s2'
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
os.makedirs(OUT, exist_ok=True)
TRAIN = FORCE_TRAIN or not os.path.exists(f'{ADAPTER}/adapter_model.safetensors')
print('train:', TRAIN, '| adapter:', ADAPTER, '| out:', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train: True | adapter: /content/drive/MyDrive/rk1_houseplant/s2_sft_v2 | out: /content/drive/MyDrive/rk1_houseplant/eval_s2


In [3]:
%cd /content
!git clone https://github.com/Schaft-s/LLM || (cd LLM && git pull)
%cd /content/LLM
import sys
sys.path.insert(0, '/content/LLM')
!wc -l data/sft_v2/sft_train.jsonl  # expect 1060
# Safe install: only our pins. Needs bitsandbytes>=0.49.2 for Colab CUDA 13.
!bash scripts/colab_install.sh
!python scripts/smoke_colab_stack.py
!git rev-parse HEAD > "{OUT}/git_commit.txt"
!mkdir -p "{OUT}/configs" && cp configs/*.yaml "{OUT}/configs/"
!nvidia-smi --query-gpu=name,memory.total --format=csv > "{OUT}/gpu.txt"

/content
Cloning into 'LLM'...
remote: Enumerating objects: 239, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 239 (delta 104), reused 201 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (239/239), 3.69 MiB | 11.84 MiB/s, done.
Resolving deltas: 100% (104/104), done.
/content/LLM
1060 data/sft_v2/sft_train.jsonl
Found existing installation: transformers 5.22.1
Uninstalling transformers-5.22.1:
  Successfully uninstalled transformers-5.22.1
Found existing installation: peft 0.21.2
Uninstalling peft-0.21.2:
  Successfully uninstalled peft-0.21.2
Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have h

## Pilot (20 steps) → reload + 1 generation

In [4]:
import subprocess

def run(cmd, log):
    # stream output to the cell and to a log on Drive; fail the cell on error
    with open(log, 'a') as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end=''); f.write(line)
    if p.wait():
        raise SystemExit(f'{cmd} failed, see {log}')

if TRAIN:
    run([sys.executable, 'scripts/train_qlora.py', '--pilot'], f'{OUT}/train_pilot.log')
    from src.train.generate import generate_chat, load_base_or_adapter
    from src.train.sft_qlora import load_yaml
    import gc, torch
    model_cfg = load_yaml('configs/model.yaml')
    model, tok = load_base_or_adapter(model_cfg, adapter_dir='artifacts/weights/pilot_qlora')
    print(generate_chat(
        model, tok,
        'My snake plant has mushy leaves at the base after weekly watering in low light. What should I check first?',
        system_prompt=model_cfg.get('system_prompt', '').strip(),
        max_new_tokens=256, temperature=0.2,
    ))
    del model; gc.collect(); torch.cuda.empty_cache()
else:
    print('adapter already on Drive → skip pilot')

I0000 00:00:1794149298.515040    3844 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794149301.465708    3844 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

Fetching 2 files: 100%|██████████| 2/2 [00:49<00:00, 24.98s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:29<00:00, 14.96s/it]

Generating train split: 0 examples [00:00, ? examples/s]
Generating train split: 1060 examples [00:00, 19389.11 examples/s]

Generating train split: 0 examples [00:00, ? examples/s]
Generating train split: 39 examples [00:00, 16585.00 examples/s]

Map: 100%|██████████| 1060/1060 [00:00<00:00, 7808.17 examples/s]

Map: 100%|██████████| 39/39 [00:00<00:00, 4349.78 examples/s]

Tokenizing train dataset: 100%|██████████| 1060/1060 [0

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The mushy leaves at the base are likely due to overwatering. Check the soil moisture level and make sure it's dry before watering again. If you're unsure about the soil moisture level, you can insert your finger into the soil to check how wet it is.


## S2: полное QLoRA на SFT v2 → копия на Drive

In [5]:
if TRAIN:
    run([sys.executable, 'scripts/train_qlora.py'], f'{OUT}/train_s2.log')
    !rm -rf "{ADAPTER}" && cp -r artifacts/weights/s2_sft_v2 "{ADAPTER}"
print(sorted(os.listdir(ADAPTER)))
assert os.path.exists(f'{ADAPTER}/adapter_model.safetensors')

I0000 00:00:1794149637.513937    5365 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

Loading checkpoint shards: 100%|██████████| 2/2 [00:20<00:00, 10.12s/it]

Tokenizing eval dataset: 100%|██████████| 39/39 [00:00<00:00, 411.89 examples/s]

Truncating eval dataset: 100%|██████████| 39/39 [00:00<00:00, 7398.03 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.

  0%|          | 0/133 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the 

## Оценка: base vs SFT

Генерация → автопроверка `no_invented_experience` → судья (OpenRouter) → правило выбора чекпоинта (судья по plant_dev ∪ garden_dev не хуже базы; general_dev не падает больше чем на 5 п.п.) → test → графики. Возобновляемо.

In [ ]:
LIMIT = '--limit 3' if SMOKE else ''
!python scripts/run_eval_pipeline.py --out-dir "{OUT}" --adapter-root "{ADAPTER}" {LIMIT} 2>&1 | tee -a "{OUT}/run.log"

[15:15:39] candidates: ['sft_final', 'sft_step50', 'sft_step100']
I0000 00:00:1794150953.360606   10833 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Loading checkpoint shards: 100%|██████████| 2/2 [00:19<00:00,  9.66s/it]
[15:16:21] == baseline samples
[15:16:21]   baseline_samples__base.jsonl: 2 to generate (0 done)
[15:17:06]     2/2 (45s/batch)
[15:17:06]   baseline_samples__sft_final.jsonl: 2 to generate (0 done)
[15:17:19]     2/2 (13s/batch)
[15:17:19] == dev / base
[15:17:19]   general_dev__base.jsonl: 80 to generate (0 done)
[15:17:20]     8/80 (1s/batch)
[15:17:21]     16/80 (1s/batch)
[15:17:22]     24/80 (1s/batch)
[15:17:23]     32/80 (1s/batch)
[15:17:24]     40/80 (1s/batch)
[15:17:26]     48/80 (2s/batch)
[15:17:28]     56/80 (2s/batch)
[15:17:29]     64/80 (1s/batch)
[15:17:30]     72/80 (1s/batch)
[15:17:31]     80/80 (1s/batch)
[15:17:31]   plant_dev__base.jsonl: 39 to generate (0 done)
[15:18:06]     8/39 (36s/batch)
[15:18:51

In [ ]:
import glob
from IPython.display import Image, Markdown, display

display(Markdown(open(f'{OUT}/summary.md').read()))
for p in sorted(glob.glob(f'{OUT}/plots/*.png')):
    print(p)
    display(Image(p))
drive.flush_and_unmount()
print('Drive flushed')